# 05 — Text classification with TF-IDF and logistic regression

A support team wants to route a short message to one of five queues. We will turn text into sparse vectors, derive what the vectorizer computes, fit a regularized multinomial logistic model, and examine the mistakes. The data are authored synthetic templates; the final score is a lesson result, not a claim about real support traffic.

## 1. Load the fixture and keep template families together

Rows from one authored template are related examples. If one sibling lands in training and another in testing, the test can reward recognition of a template rather than generalization to a new one. We therefore assign whole groups within each category. The rule is fixed before fitting: 8 groups per category for training, 2 for validation, and 2 for the final test. Each group contains eight rows, so every split is balanced.

In [ ]:
from pathlib import Path
import random
import pandas as pd

for candidate in (Path.cwd(), *Path.cwd().parents):
    data_path = candidate / "data" / "support_tickets.csv"
    if data_path.exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Run this notebook from the repository or one of its subfolders."
    )

tickets = pd.read_csv(data_path)
print(
    f"Loaded {len(tickets)} tickets from {tickets['template_group'].nunique()} template groups."
)
tickets.head(3)

In [ ]:
required = {"ticket_id", "category", "template_group", "text"}
assert required <= set(tickets.columns)
assert tickets[list(required)].notna().all().all()
assert tickets.groupby("template_group")["category"].nunique().max() == 1
assert tickets.groupby("template_group").size().nunique() == 1

categories = sorted(tickets["category"].unique())
rng = random.Random(2026)
split_groups = {"train": [], "validation": [], "test": []}
for category in categories:
    groups = sorted(
        tickets.loc[tickets.category == category, "template_group"].unique()
    )
    rng.shuffle(groups)
    split_groups["train"].extend(groups[:8])
    split_groups["validation"].extend(groups[8:10])
    split_groups["test"].extend(groups[10:12])

splits = {
    name: tickets[tickets.template_group.isin(group_ids)].reset_index(drop=True)
    for name, group_ids in split_groups.items()
}
assert set(split_groups["train"]).isdisjoint(split_groups["validation"])
assert set(split_groups["train"]).isdisjoint(split_groups["test"])
assert set(split_groups["validation"]).isdisjoint(split_groups["test"])
for name, frame in splits.items():
    counts = frame.category.value_counts().reindex(categories, fill_value=0)
    assert (counts > 0).all() and counts.nunique() == 1
    print(
        f"{name:10s}: {len(frame):3d} rows; "
        + ", ".join(f"{k}={v}" for k, v in counts.items())
    )

## 2. TF-IDF as a weighted sparse vector

Fix a vocabulary of $p$ terms and let $x_i\in\mathbb{R}^p$ contain the raw count of each term in document $i$. For term $j$, document frequency is $df_j=\#\{i:x_{ij}>0\}$. With scikit-learn's default `smooth_idf=True`, the inverse document frequency is

$$\operatorname{idf}_j=\log\!\left(\frac{1+n}{1+df_j}\right)+1.$$

The added one in numerator and denominator smooths document frequency. A term in every fitted document has weight one. The learned vocabulary contains only terms observed in the fitting corpus, so a genuinely unseen term is not assigned a column. The default word analyzer lowercases text, uses unigrams, and keeps tokens matching `(?u)\b\w\w+\b` (Unicode word characters, at least two characters). It uses no stop-word list or accent stripping. The default `sublinear_tf=False` leaves term frequency as the raw count. `TfidfVectorizer` multiplies count by IDF and, by default, applies L2 normalization to each row. A nonempty document vector $v$ becomes $v/\|v\|_2$; its coordinates therefore have sum of squares one.

With L2-normalized rows $u$ and $v$, cosine similarity is

$$\cos(u,v)=\frac{u^Tv}{\|u\|_2\|v\|_2}=u^Tv.$$

The representation is sparse because any document uses only a small fraction of all vocabulary terms. We will inspect the exact fitted convention. The vocabulary and IDF must be learned from training text only; putting the vectorizer inside a pipeline enforces that rule during fitting.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X_train_preview = vectorizer.fit_transform(splits["train"].text)
print("shape:", X_train_preview.shape)
print("sparse type:", type(X_train_preview).__name__)
print(
    "density:",
    X_train_preview.nnz / (X_train_preview.shape[0] * X_train_preview.shape[1]),
)
print(
    "norms of first three rows:",
    (X_train_preview[:3].multiply(X_train_preview[:3]).sum(axis=1).A1) ** 0.5,
)
print(
    {
        name: getattr(vectorizer, name)
        for name in [
            "lowercase",
            "strip_accents",
            "ngram_range",
            "stop_words",
            "token_pattern",
            "use_idf",
            "smooth_idf",
            "sublinear_tf",
            "norm",
        ]
    }
)

### One worked IDF calculation

Suppose a vocabulary term occurs in 2 of 8 fitted documents. The smoothed IDF is $\log(9/3)+1=\log 3+1$. A ubiquitous term has $df=8$ and receives $\log(9/9)+1=1$. Run the cell to check scikit-learn's stored value for the first case.

In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

documents = ["common rare", "common rare"] + ["common"] * 6
idf_vectorizer = TfidfVectorizer()
idf_vectorizer.fit(documents)
idf_by_term = dict(zip(idf_vectorizer.get_feature_names_out(), idf_vectorizer.idf_))
n_documents = len(documents)
document_frequency = sum("rare" in document.split() for document in documents)
manual_idf = np.log((1 + n_documents) / (1 + document_frequency)) + 1
print("manual smoothed IDF:", manual_idf)
print("scikit-learn stored IDF:", idf_by_term["rare"])
print("ubiquitous term IDF:", idf_by_term["common"])
assert np.isclose(manual_idf, idf_by_term["rare"])
assert np.isclose(idf_by_term["common"], 1.0)

## 3. Multinomial logistic regression

For a document vector $x_i$ and $K$ categories, assign a coefficient vector $w_k$ and intercept $b_k$ to each category. Its score is $z_{ik}=w_k^Tx_i+b_k$. Softmax converts scores to probabilities:

$$p_{ik}=\frac{e^{z_{ik}}}{\sum_{r=1}^{K}e^{z_{ir}}}.$$

For one-hot labels $y_{ik}$, average cross-entropy is

$$L(W,b)=-\frac{1}{n}\sum_i\sum_k y_{ik}\log p_{ik}.$$

Differentiating the softmax-cross-entropy composition gives $\nabla_{w_k}L=\frac{1}{n}\sum_i(p_{ik}-y_{ik})x_i$. Scikit-learn's L2-regularized objective also penalizes large coefficient vectors; in `LogisticRegression`, larger `C` means weaker regularization. The `lbfgs` solver handles the multiclass softmax objective. We set `max_iter` high enough for this sparse, small problem and inspect convergence as needed.

Softmax probabilities are unchanged if the same coefficient vector and intercept are added to every category, so a single class coefficient has no absolute interpretation. The pairwise log-odds are `log(p_k/p_l) = (w_k - w_l)^T x + (b_k - b_l)`. Correlated terms and regularization affect these contrasts, which describe the fitted model rather than causes.

## 4. Select regularization on validation, then test once

Macro-F1 computes an F1 score separately for every category, then averages those five scores equally. It is useful when categories deserve equal attention, while ordinary accuracy weights rows. We compare a short, declared list of `C` values on validation only. After choosing `C`, we refit the same pipeline on training plus validation and evaluate the held-out test set once. We also compare a `DummyClassifier(strategy="most_frequent")` fitted on train plus validation. It ignores text and supplies a simple reference on the same frozen test rows.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline

X_train = splits["train"].text
y_train = splits["train"].category
X_validation = splits["validation"].text
y_validation = splits["validation"].category

candidate_C = [0.05, 0.2, 1.0, 5.0, 20.0]
validation_rows = []
for C in candidate_C:
    candidate = Pipeline(
        [
            ("tfidf", TfidfVectorizer()),
            ("logistic", LogisticRegression(C=C, solver="lbfgs", max_iter=1500)),
        ]
    )
    candidate.fit(X_train, y_train)
    prediction = candidate.predict(X_validation)
    validation_rows.append(
        {"C": C, "macro_f1": f1_score(y_validation, prediction, average="macro")}
    )

validation_scores = pd.DataFrame(validation_rows)
display(validation_scores)
best_C = float(
    validation_scores.sort_values(["macro_f1", "C"], ascending=[False, True]).iloc[0].C
)
print("Selected C using validation macro-F1:", best_C)

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

train_validation = pd.concat([splits["train"], splits["validation"]], ignore_index=True)
final_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer()),
        ("logistic", LogisticRegression(C=best_C, solver="lbfgs", max_iter=1500)),
    ]
)
final_model.fit(train_validation.text, train_validation.category)

# This is the one final test evaluation after model selection is frozen.
test_prediction = final_model.predict(splits["test"].text)
test_macro_f1 = f1_score(splits["test"].category, test_prediction, average="macro")
print("test accuracy:", accuracy_score(splits["test"].category, test_prediction))
print("test macro-F1:", test_macro_f1)

baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(train_validation.text, train_validation.category)
baseline_prediction = baseline.predict(splits["test"].text)
print(
    "most-frequent baseline accuracy:",
    accuracy_score(splits["test"].category, baseline_prediction),
)
print(
    "most-frequent baseline macro-F1:",
    f1_score(splits["test"].category, baseline_prediction, average="macro"),
)
print(
    classification_report(
        splits["test"].category, test_prediction, labels=categories, zero_division=0
    )
)
confusion = pd.DataFrame(
    confusion_matrix(splits["test"].category, test_prediction, labels=categories),
    index=[f"actual: {c}" for c in categories],
    columns=[f"predicted: {c}" for c in categories],
)
display(confusion)

## 5. Read mistakes and coefficients

A confusion matrix counts actual rows in its rows and predictions in its columns. Off-diagonal cells identify category pairs worth inspecting. Error review asks whether a ticket really fits one queue, whether a phrase is ambiguous, and whether the model relies on a shared template cue. The synthetic test examples remain related to the same small set of authored situations, even though their groups were withheld.

In [ ]:
test_review = splits["test"][["ticket_id", "template_group", "category", "text"]].copy()
test_review["predicted"] = test_prediction
errors = test_review[test_review.category != test_review.predicted].copy()
print(f"{len(errors)} errors among {len(test_review)} held-out rows.")
if errors.empty:
    print("There are no test errors to inspect in this run.")
else:
    errors["actual/predicted"] = errors.category + " -> " + errors.predicted
    display(
        errors[["ticket_id", "template_group", "actual/predicted", "text"]].head(12)
    )

In [ ]:
fitted_tfidf = final_model.named_steps["tfidf"]
fitted_logistic = final_model.named_steps["logistic"]
feature_names = fitted_tfidf.get_feature_names_out()

for category_index, category in enumerate(fitted_logistic.classes_):
    weights = fitted_logistic.coef_[category_index]
    highest = np.argsort(weights)[-8:][::-1]
    lowest = np.argsort(weights)[:5]
    print(f"\n{category}: terms most associated with its logit")
    print([(feature_names[i], round(weights[i], 3)) for i in highest])
    print(
        "terms lowering its logit:",
        [(feature_names[i], round(weights[i], 3)) for i in lowest],
    )

## 6. What this lesson does and does not establish

TF-IDF plus a linear classifier is a clear supervised baseline: it is fast, sparse, inspectable, and often a useful first comparison. A pretrained embedding model uses dense vectors designed to encode semantic relationships, which may help match paraphrases with few shared words; performance depends on the model, domain, and evaluation. Embeddings do not remove the need for labels, group-aware evaluation, or error review when the task is supervised routing; adapting or scoring a model still needs a defensible evaluation. This course uses no external model and downloads no pretrained weights.

Our split tests new authored template families from the same generator. It prevents sibling leakage but cannot reveal performance on naturally written, evolving, multilingual, imbalanced, or sensitive customer messages. A real deployment study would require appropriately governed representative data, a time-aware or source-aware evaluation, operational costs, and human review.